# Advanced lab 4 — MLflow tracing, GenAI evaluation, and regression evidence

Use MLflow as the AI engineering evidence plane: traces, reviewed datasets, custom and built-in GenAI scorers, baseline/change comparisons, prompt lineage, and release decisions. Use `mlflow.genai.evaluate()`; do not mix these scorers with classic `mlflow.models.evaluate()` metrics.

Current references: [MLflow GenAI evaluation](https://mlflow.org/docs/latest/genai/eval-monitor/running-evaluation/), [custom scorers](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/custom/), [trace evaluation](https://mlflow.org/docs/latest/genai/eval-monitor/running-evaluation/traces/), and [evaluation datasets](https://mlflow.org/docs/latest/genai/datasets/).

In [ ]:
import importlib.util
import json
import math
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / 'examples' / 'foundry-curriculum')
    if (candidate / 'notebook_setup.py').is_file()
)
spec = importlib.util.spec_from_file_location(
    'foundry_curriculum_setup', curriculum_root / 'notebook_setup.py'
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

In [ ]:
starter_cases = [
    json.loads(line)
    for line in (curriculum_root / 'data' / 'evaluation_cases.jsonl')
    .read_text(encoding='utf-8')
    .splitlines()
    if line.strip()
]
evaluation_records = [
    {
        'inputs': {'request': case['input']},
        'expectations': {
            **case['expectations'],
            'case_id': case['case_id'],
            'category': case['category'],
        },
    }
    for case in starter_cases
]
assert all(set(record) == {'inputs', 'expectations'} for record in evaluation_records)
assert len(evaluation_records) == 20
evaluation_records[:2]

In [ ]:
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer


def output_text(outputs) -> str:
    if isinstance(outputs, dict):
        return str(outputs.get('text', ''))
    return str(outputs)


@scorer(name='required_terms', aggregations=['mean'])
def required_terms(outputs, expectations) -> Feedback:
    text = output_text(outputs).lower()
    required = [str(term).lower() for term in expectations.get('must_cover', [])]
    missing = [term for term in required if term not in text]
    return Feedback(
        name='required_terms',
        value=not missing,
        rationale=(
            'All required concepts were present.'
            if not missing
            else f'Missing required concepts: {missing}'
        ),
    )


@scorer(name='critical_data_policy', aggregations=['mean'])
def critical_data_policy(outputs, expectations) -> Feedback:
    text = output_text(outputs).lower()
    forbidden = ('bearer token', 'api key:', 'client secret:', 'password=')
    leaked = [marker for marker in forbidden if marker in text]
    return Feedback(
        name='critical_data_policy',
        value=not leaked,
        rationale='No forbidden marker found.' if not leaked else f'Found {leaked}',
    )


required_terms({'text': 'Use Entra ID with Azure CLI.'}, {'must_cover': ['Entra ID']})

In [ ]:
RUN_MLFLOW_EVAL = False
RUN_CONNECTED = False


def predict_foundry(request: str) -> dict[str, str]:
    if not RUN_CONNECTED:
        raise RuntimeError('Set RUN_CONNECTED=True to score the configured agent.')
    _conversation, response = helpers.create_agent_response(
        session, request, allow_network=True
    )
    return {'text': response.output_text}


def enforce_metric_gate(metrics: dict[str, float], thresholds: dict[str, float]):
    missing = set(thresholds) - set(metrics)
    if missing:
        raise RuntimeError(f'Missing MLflow metrics: {sorted(missing)}')
    failed = []
    for name, threshold in thresholds.items():
        value = float(metrics[name])
        if math.isnan(value) or value < threshold:
            failed.append(f'{name}={value:.3f} < {threshold:.3f}')
    if failed:
        raise RuntimeError('MLflow evaluation gate failed: ' + '; '.join(failed))


if RUN_MLFLOW_EVAL:
    import mlflow

    mlflow.set_experiment(session.context.settings.effective_experiment_name)
    mlflow_result = mlflow.genai.evaluate(
        data=evaluation_records,
        predict_fn=predict_foundry,
        scorers=[required_terms, critical_data_policy],
    )
    enforce_metric_gate(
        mlflow_result.metrics,
        {'required_terms/mean': 0.85, 'critical_data_policy/mean': 1.0},
    )
    print({'run_id': mlflow_result.run_id, 'metrics': mlflow_result.metrics})
else:
    print(
        'MLflow evaluation skipped; both MLflow and Foundry calls '
        'are explicit opt-ins.'
    )

## Add judges deliberately

After deterministic checks work, add built-in GenAI scorers such as relevance, safety, groundedness, or tool-call correctness for the applicable slice. Configure a judge deployment, limit `MLFLOW_GENAI_EVAL_MAX_WORKERS` to protect quota, calibrate judges against human labels, and retain the human review separately. Never let an evaluator exception or missing score count as a pass.

In [ ]:
RUN_MLFLOW_TRACE = False

if RUN_MLFLOW_TRACE:
    import mlflow
    from mlflow.entities import SpanType

    mlflow.set_experiment(session.context.settings.effective_experiment_name)
    with mlflow.tracing.context(session_id='synthetic-conversation-001'):
        with mlflow.start_span(
            name='context.retrieve', span_type=SpanType.RETRIEVER
        ) as span:
            span.set_inputs({'case_id': 'context-provenance-01'})
            span.set_outputs(
                [
                    {
                        'page_content': 'Synthetic approved policy excerpt.',
                        'doc_uri': 'uc://governance/release-policy',
                        'chunk_id': 'release-7',
                        'metadata': {
                            'classification': 'synthetic',
                            'freshness': 'current',
                        },
                    }
                ]
            )
    mlflow.flush_trace_async_logging()
    print({'trace_id': mlflow.get_last_active_trace_id()})
else:
    print('Manual MLflow trace skipped; the example captures synthetic content only.')

In [ ]:
from mlflow.entities import SpanType, Trace


@scorer(name='subagent_routing', aggregations=['mean'])
def subagent_routing(trace: Trace, expectations: dict) -> Feedback:
    actual = [
        span.name for span in trace.search_spans(span_type=SpanType.AGENT)
    ]
    expected = expectations['agent_trajectory']
    return Feedback(
        name='subagent_routing',
        value=actual == expected,
        rationale=f'Expected {expected}; observed {actual}',
    )


RUN_HISTORICAL_TRACE_EVAL = False
if RUN_HISTORICAL_TRACE_EVAL:
    import mlflow

    mlflow.set_experiment(session.context.settings.effective_experiment_name)
    historical = mlflow.search_traces(
        filter_string="attributes.status = 'OK'"
    )
    historical_result = mlflow.genai.evaluate(
        data=historical, scorers=[subagent_routing]
    )
    print({'run_id': historical_result.run_id})
else:
    print('Historical trace evaluation skipped.')

In [ ]:
lineage_plan = {
    'application_version': session.context.settings.resource.release,
    'agent_name': session.labs.agent.name,
    'agent_version': session.labs.agent.version,
    'context_policy_version': 'context-policy-v1',
    'prompt_version': 'immutable-registry-version',
    'dataset_version': 'foundry-curriculum-eval-v1',
    'decision_vocabulary': ['adopt', 'reject', 'inconclusive'],
    'sequence': 'baseline -> change -> result -> decision',
}
lineage_plan

## Exit criteria

Run the same versioned records against baseline and change, examine failures by risk slice, and record an adopt/reject/inconclusive decision. Preserve model, prompt, tool, index, embedding, chunking, context-policy, dataset, and scorer versions. Production monitoring samples traces; offline regression evaluation protects the release gate.